In [ ]:
!pip install jupyterplot
!pip install pyserial
!pip install git+https://github.com/lbaitemple/Proto-Grid.git@dev
!pip install jupyterlab-executor

In [ ]:
!pip install revidyne
!pip show revidyne

In [ ]:
!mkdir -p streaming

In [ ]:
%%writefile streaming/main.py

"""
Import libraries
"""

from revidyne import AllDevice
from bokeh import models, plotting, io
from bokeh.io import show, output_notebook
from bokeh.layouts import column, row
import pandas as pd
from itertools import cycle
from datetime import datetime
import time, math, sys, re, os
from contextlib import contextmanager
import io as console_io
import numpy as np
import heapq
import random

"""
PID controller implementation (https://medium.com/@aleksej.gudkov/python-pid-controller-example-a-complete-guide-5f35589eec86)
"""

class PIDController:
    def __init__(self, Kp, Ki, Kd, setpoint):
        self.Kp = Kp
        self.Ki = Ki
        self.Kd = Kd
        self.setpoint = setpoint
        self.previous_error = 0
        self.integral = 0

    def compute(self, process_variable, dt):
        # Calculate error
        error = self.setpoint - process_variable
        
        # Proportional term
        P_out = self.Kp * error
        
        # Integral term
        self.integral += error * dt
        I_out = self.Ki * self.integral
        
        # Derivative term
        derivative = (error - self.previous_error) / dt
        D_out = self.Kd * derivative
        
        # Compute total output
        output = P_out + I_out + D_out
        
        # Update previous error
        self.previous_error = error
        
        return output

"""
Initialie the respective devices - baud rate 115200
"""

a=AllDevice(115200)
mm=a.getAllDevice()
time.sleep(8)
for key in mm.keys():
    mm[key].getDevice().call('init')
    time.sleep(0.2)
time.sleep(8)

"""
Global variables
"""

i = 0
c = 0
outKWGens = []
outKWSols = []
outKWWTs = []
outVGens = []
outVSols = []
outVWTs = []
co2accum = None
demand = None
surplus = None
needle = None
window_universal = 120
dial_min = -1500
dial_max = 1500
gen_max = 250
pq_map = set()
pq_demand = []
pq_supply = []
power_demand = 0
power_supply = 0
pid_iter0 = 0
enable_program_pid = 0
game = 0

"""
Method to suppress outputs associated with commands sent to devices
"""

@contextmanager
def suppress_stdout():
    with open(os.devnull, "w") as devnull:
        old_stdout = sys.stdout
        sys.stdout = devnull
        try:  
            yield
        finally:
            sys.stdout = old_stdout


"""
Method is triggered every 2 secs and updates the streamed values for each of the devices to their respective plots.
It is also used to execute the experiments at the dicrete intervals of time.
    1. Genetic algorithm based PID value determination - both using PID settings of the generators and the controllers.
    2. Grid game simulation for one player
"""
            
def update(event=None):
    global i, c, outKWGens, outKWSols, outKWWTs, co2accum, outVGens, outVSols, outVWTs
    global demand, surplus, needle, power_demand, power_supply
    i+=1
    totalSupply = 0
    totalDemand = 0
    solarSupply = 0
    windSupply = 0

    # Plot power output and voltage output for generators. Labels are inside the plots.
    
    for j, gen in enumerate([x for x in mm.keys() if x.startswith('generator')]):
        with suppress_stdout():
            outLoad = 0
            try:
                outLoad = float(mm[gen].getDevice().call('getKW')[0])
            except:
                pass
        outLoad = 0 if math.isnan(outLoad) else outLoad
        totalSupply += outLoad
        new_data={'Time': [i], 'OutputLoad': [outLoad]}
        outKWGens[j].stream(new_data, rollover=window_universal)
        with suppress_stdout():
            outVolt = 0
            try:
                outVolt = float(mm[gen].getDevice().call('getVal')[0])
            except:
                pass
        outVolt = 0 if math.isnan(outVolt) else outVolt
        new_data={'Time': [i], 'VoltageOutput': [outVolt]}
        outVGens[j].stream(new_data, rollover=window_universal)
        
        # Generators are considered carbon based and hence accumulated CO2 emissions are plotted
        
        with suppress_stdout():
            outCO2 = 0
            try:
                outCO2 = float(mm[gen].getDevice().call('getCarbon')[0])
            except:
                pass
        outCO2 = 0 if math.isnan(outCO2) else outCO2
        c += outCO2

    # Plot power output and voltage output for solar panels. Labels are inside the plots.
    
    for j, sol in enumerate([x for x in mm.keys() if x.startswith('solar')]):
        with suppress_stdout():
            outLoad = 0
            try:
                outLoad = float(mm[sol].getDevice().call('getKW')[0])
            except:
                pass
        outLoad = 0 if math.isnan(outLoad) else outLoad
        totalSupply += outLoad
        solarSupply += outLoad
        new_data={'Time': [i], 'OutputLoad': [outLoad]}
        outKWSols[j].stream(new_data, rollover=window_universal)
        with suppress_stdout():
            outVolt = 0
            try:
                outVolt = float(mm[sol].getDevice().call('getVal')[0])
            except:
                pass
        outVolt = 0 if math.isnan(outVolt) else outVolt
        new_data={'Time': [i], 'VoltageOutput': [outVolt]}
        outVSols[j].stream(new_data, rollover=window_universal)
        with suppress_stdout():
            outCO2 = 0
            try:
                outCO2 = float(mm[sol].getDevice().call('getCarbon')[0])
            except:
                pass
        outCO2 = 0 if math.isnan(outCO2) else outCO2
        c += outCO2

    # Plot power output and voltage output for wind turbines. Labels are inside the plots.
    
    for j, wind in enumerate([x for x in mm.keys() if x.startswith('wind')]):
        with suppress_stdout():
            outLoad = 0
            try:
                outLoad = float(mm[wind].getDevice().call('getKW')[0])
            except:
                pass
        outLoad = 0 if math.isnan(outLoad) else outLoad
        totalSupply += outLoad
        windSupply += outLoad
        new_data={'Time': [i], 'OutputLoad': [outLoad]}
        outKWWTs[j].stream(new_data, rollover=window_universal)
        with suppress_stdout():
            outVolt = 0
            try:
                outVolt = float(mm[wind].getDevice().call('getVal')[0])
            except:
                pass
        outVolt = 0 if math.isnan(outVolt) else outVolt
        new_data={'Time': [i], 'VoltageOutput': [outVolt]}
        outVWTs[j].stream(new_data, rollover=window_universal)
        with suppress_stdout():
            outCO2 = 0
            try:
                outCO2 = float(mm[wind].getDevice().call('getCarbon')[0])
            except:
                pass
        outCO2 = 0 if math.isnan(outCO2) else outCO2
        c += outCO2

    # Plot combined power consumed by all the houseload devices - the houseloads have 4 lights each, which can be individually turned on/off
    
    for j, house in enumerate([x for x in mm.keys() if x.startswith('houseload')]):
        outLoad = household_get_call(mm[house].getDevice(), 'getLoadVal')
        totalDemand += outLoad

    # Plot the surplus / deficit power in KW into an analog dial visualization.

    new_data={'Time': [i], 'HouseLoads': [totalDemand]}
    demand.stream(new_data, rollover=window_universal)
    new_data={'Time': [i], 'CO2Accumulation': [c]}
    co2accum.stream(new_data, rollover=window_universal)
    power_demand = totalDemand
    power_supply = totalSupply
    s = totalSupply - totalDemand
    current_data_surplus = dict(surplus.data)
    current_data_surplus['labels']=[str("{:.1f}".format(s))]
    surplus.data = current_data_surplus
    current_data_needle = dict(needle.data)
    needle_angle = np.pi - ((s - dial_min) / (dial_max - dial_min)) * np.pi
    x = 0.8 * np.cos(needle_angle)
    y = 0.8 * np.sin(needle_angle)
    current_data_needle['x']=[x]
    current_data_needle['y']=[y]
    current_data_needle['angle']=[needle_angle]
    needle.data = current_data_needle

    # Show PID experiment for controlling generator power - both based on device' internal pid control as well as programmatically 
    
    global pid_iter0
    if pid_iter0 >= 1 and pid_iter0 <= 11:
        apply_fitness_PID_device('generator0', int(setGenLoad.value))
    if pid_iter0 == 11:
        pid_iter0 = 0

    global enable_program_pid
    if enable_program_pid == 1:
        apply_fitness_PID_program('generator1', int(setGenLoad.value))

    # Show grid game experiment for single player - display utility value
    play_grid_game (windSupply, solarSupply)


"""
Method to display a dial needle kind of widget showing surplus / deficit power in KW into an analog dial visualization
This is the initial plot setup
"""

def create_dial():
    global surplus, needle
    z = plotting.figure(width=300, height=300, title="Power Surplus (kW)")
    z.toolbar_location = None
    z.x_range = models.Range1d(start=-1.2, end=1.2)
    z.y_range = models.Range1d(start=-0.4, end=1.2)
    z.axis.visible = False
    z.grid.visible = False
    z.annular_wedge(x = 0, y = 0, inner_radius = 0.8, outer_radius = 1, start_angle=2 * np.pi / 3, end_angle=np.pi, fill_color="red")
    z.annular_wedge(x = 0, y = 0, inner_radius = 0.8, outer_radius = 1, start_angle=np.pi / 3, end_angle=2 * np.pi / 3, fill_color="orange")
    z.annular_wedge(x = 0, y = 0, inner_radius = 0.8, outer_radius = 1, start_angle=0, end_angle=np.pi / 3, fill_color="lightgreen")
    
    needle = models.ColumnDataSource(data=dict(x=[0], y=[0.8], angle=[0]))
    z.add_layout(models.Arrow(x_start=0, y_start=0, x_end="x", y_end="y", line_width=2, line_color="black", 
                            source=needle, end=models.VeeHead(size=5)))
    
    surplus = models.ColumnDataSource(dict(x=[0], y=[-0.1], labels=[str(0)]))
    z.add_layout(models.LabelSet(x='x', y='y', text='labels', source=surplus, text_align='center', 
                            text_font_size='10pt', level='glyph'))
    return z

"""
Method to display plot for all the generators along with legends - power and voltage
This is the initial plot setup
"""

def plotGenerators():
    global i, outKWGens, outVGens
    colors = ["red", "blue", "green", "pink", "brown"]
    for j, gen in enumerate([x for x in mm.keys() if x.startswith('generator')]):
        outKWGens.append(models.ColumnDataSource(data=dict(Time=[i], OutputLoad=[0])))
        outVGens.append(models.ColumnDataSource(data=dict(Time=[i], VoltageOutput=[0])))
        p.line(x="Time", y="OutputLoad", source=outKWGens[j], width=2, color=colors[j], legend_label=gen)
        pv.line(x="Time", y="VoltageOutput", source=outVGens[j], width=2, color=colors[j], legend_label=gen)

"""
Method to display plot for all the solar panels along with legends - power and voltage
This is the initial plot setup
"""

def plotSolars():
    global i, outKWSols, outVSols
    colors = ["red", "blue", "green", "pink", "brown"]
    for j, gen in enumerate([x for x in mm.keys() if x.startswith('solar')]):
        outKWSols.append(models.ColumnDataSource(data=dict(Time=[i], OutputLoad=[0])))
        outVSols.append(models.ColumnDataSource(data=dict(Time=[i], VoltageOutput=[0])))
        q.line(x="Time", y="OutputLoad", source=outKWSols[j], width=2, color=colors[j], legend_label=gen)
        qv.line(x="Time", y="VoltageOutput", source=outVSols[j], width=2, color=colors[j], legend_label=gen)

"""
Method to display plot for all the wind turbines along with legends - power and voltage
This is the initial plot setup
"""

def plotWindTurbines():
    global i, outKWWTs, outVWTs
    colors = ["red", "blue", "green", "pink", "brown"]
    for j, gen in enumerate([x for x in mm.keys() if x.startswith('wind')]):
        outKWWTs.append(models.ColumnDataSource(data=dict(Time=[i], OutputLoad=[0])))
        outVWTs.append(models.ColumnDataSource(data=dict(Time=[i], VoltageOutput=[0])))
        r.line(x="Time", y="OutputLoad", source=outKWWTs[j], width=2, color=colors[j], legend_label=gen)
        rv.line(x="Time", y="VoltageOutput", source=outVWTs[j], width=2, color=colors[j], legend_label=gen)

"""
Method to display plot for accumulated CO2 emissions from the generators
This is the initial plot setup
"""

def plotDemandAndCO2():
    global i, demand, co2accum
    demand = models.ColumnDataSource(data=dict(Time=[i], HouseLoads=[0]))
    co2accum = models.ColumnDataSource(data=dict(Time=[i], CO2Accumulation=[0]))
    
    s.line(x="Time", y="HouseLoads", source=demand, width=2)
    t.line(x="Time", y="CO2Accumulation", source=co2accum, width=2)

"""
Method to display plot for total power consumed by houseloads
This is the initial plot setup
"""

def household_get_call(dev, cmd):
    output_capture = console_io.StringIO()
    original_stdout = sys.stdout
    sys.stdout = output_capture
    try:
        dev.call(cmd)
    except:
        pass
    finally:
        sys.stdout = original_stdout
    captured_output = 0.0
    try:
        captured_output = float(re.sub("[^0-9\.]", "", output_capture.getvalue().split(':')[1]))
    except:
        captured_output = 0.0
    output_capture.close()
    powerOut = captured_output
    if powerOut != 0:
        powerOut = 200 - powerOut
    return powerOut

"""
Method to send the commands from the respective devices once the command has been selected from the dropdown of the device
"""

def button_handler_apply(key1, cmd1, val1, but1):
    def apply_handler():
        dev = mm[key1].getDevice()
        plain_text = re.sub(r'<.*?>', '', cmd1.text)
        if plain_text.startswith('set'):
            new_row = {'Device': [key1], 'Command': [plain_text], 'Value': [val1.value]}
            cmds.stream(new_row, rollover=window_universal)
            if plain_text == 'setLoad':
                getattr(dev, plain_text)(int(val1.value))
            else:
                getattr(dev, plain_text)(float(val1.value))
        elif plain_text.startswith('get'):
            getVal = 0
            if key1.startswith('houseload'):
                getVal = household_get_call(dev, plain_text)
            else:
                with suppress_stdout():
                    getVal = dev.call(plain_text)[0]
            new_row = {'Device': [key1], 'Command': [plain_text], 'Value': [getVal]}
            cmds.stream(new_row, rollover=window_universal)
        elif plain_text != "":
            dev.call(plain_text)
            new_row = {'Device': [key1], 'Command': [plain_text], 'Value': [""]}
            cmds.stream(new_row, rollover=window_universal)    
        but1.disabled=True
    return apply_handler

"""
Method to demonstrate selection of PID values of generator based on genetic alogorithms - selection and crosover. 
The fitness function is taken as the percentage error from the set point.
https://medium.com/@maohar502/pid-tuning-using-machine-learning-6cf6f7fe5690
The set commands for the P, I and D values for the generators is used and the real output response is tested.
"""

def apply_fitness_PID_device (gen, setPoint):
    global pid_iter0

    def getLoad (gen):
        with suppress_stdout():
            outLoad = 0
            try:
                outLoad = float(mm[gen].getDevice().call('getKW')[0])
            except:
                pass
        outLoad = 0 if math.isnan(outLoad) else outLoad
        return outLoad

    # Iteration 1
    
    if pid_iter0 == 1:
        apply_fitness_PID_device.Kp = [0.4,0.8,1.2,1.6,2.0]
        apply_fitness_PID_device.Kd = [20.0,22.0,25.0,28.0,30.0]
        apply_fitness_PID_device.Ki = [0.1,0.5,1.0,2.0,4.0]
        apply_fitness_PID_device.e = [0] * 5
        random.shuffle(apply_fitness_PID_device.Kp)
        random.shuffle(apply_fitness_PID_device.Kd)
        random.shuffle(apply_fitness_PID_device.Ki)
        apply_fitness_PID_device.map1 = [{"Kp": kp, "Kd": kd, "Ki": ki, "e": er} 
                    for kp, kd, ki, er in zip(apply_fitness_PID_device.Kp, apply_fitness_PID_device.Kd, 
                                              apply_fitness_PID_device.Ki, apply_fitness_PID_device.e)]
        getattr(mm[gen].getDevice(), 'setKp')(apply_fitness_PID_device.map1[0]["Kp"])
        time.sleep(0.2)
        getattr(mm[gen].getDevice(), 'setKd')(apply_fitness_PID_device.map1[0]["Kd"])
        time.sleep(0.2)
        getattr(mm[gen].getDevice(), 'setKi')(apply_fitness_PID_device.map1[0]["Ki"])
        time.sleep(0.2)
        print ("At step", str(pid_iter0), "PID values and error", str(apply_fitness_PID_device.map1))
        pid_iter0 += 1
        return

    curr = getLoad(gen)
    apply_fitness_PID_device.map1[(pid_iter0-2)%5]["e"] = abs(curr - setPoint)/setPoint

    # Iteration 2
    
    if pid_iter0 == 6:
        # Sort the P, I and D combinations based on minimum errors and apply selection (1st row) and crossover (2nd and 3rd row)
        apply_fitness_PID_device.map1.sort(key=lambda x: x["e"])
        t2 = apply_fitness_PID_device.map1[1]["Kd"]
        t3 = apply_fitness_PID_device.map1[1]["Ki"]
        apply_fitness_PID_device.map1[1]["Kd"] = apply_fitness_PID_device.map1[2]["Kd"]
        apply_fitness_PID_device.map1[1]["Ki"] = apply_fitness_PID_device.map1[2]["Ki"]
        apply_fitness_PID_device.map1[2]["Kd"] = t2
        apply_fitness_PID_device.map1[2]["Ki"] = t3
        t1 = apply_fitness_PID_device.map1[3]["Kp"]
        apply_fitness_PID_device.map1[3]["Kp"] = apply_fitness_PID_device.map1[1]["Kp"]
        apply_fitness_PID_device.map1[4]["Kp"] = t1
        apply_fitness_PID_device.map1[4]["Kd"] = t2
        apply_fitness_PID_device.map1[4]["Ki"] = t3
        getattr(mm[gen].getDevice(), 'setKp')(apply_fitness_PID_device.map1[0]["Kp"])
        time.sleep(0.2)
        getattr(mm[gen].getDevice(), 'setKd')(apply_fitness_PID_device.map1[0]["Kd"])
        time.sleep(0.2)
        getattr(mm[gen].getDevice(), 'setKi')(apply_fitness_PID_device.map1[0]["Ki"])
        time.sleep(0.2)
        print ("At step", str(pid_iter0), "PID values and error", str(apply_fitness_PID_device.map1))
        pid_iter0 += 1
        return

    # Final selection
        
    if pid_iter0 == 11:
        apply_fitness_PID_device.map1.sort(key=lambda x: x["e"])
        getattr(mm[gen].getDevice(), 'setKp')(apply_fitness_PID_device.map1[0]["Kp"])
        time.sleep(0.2)
        getattr(mm[gen].getDevice(), 'setKd')(apply_fitness_PID_device.map1[0]["Kd"])
        time.sleep(0.2)
        getattr(mm[gen].getDevice(), 'setKi')(apply_fitness_PID_device.map1[0]["Ki"])
        time.sleep(0.2)
        return

    # Physically set the PID values at each iterations
    
    getattr(mm[gen].getDevice(), 'setKp')(apply_fitness_PID_device.map1[(pid_iter0-1)%5]["Kp"])
    time.sleep(0.2)
    getattr(mm[gen].getDevice(), 'setKd')(apply_fitness_PID_device.map1[(pid_iter0-1)%5]["Kd"])
    time.sleep(0.2)
    getattr(mm[gen].getDevice(), 'setKi')(apply_fitness_PID_device.map1[(pid_iter0-1)%5]["Ki"])
    time.sleep(0.2)
    print ("At step", str(pid_iter0), "PID values and error", str(apply_fitness_PID_device.map1))
    pid_iter0 += 1

"""
Method to demonstrate selection of PID values of generator based on genetic alogorithms - selection and crosover. 
The fitness function is taken as the percentage error from the set point.
https://medium.com/@maohar502/pid-tuning-using-machine-learning-6cf6f7fe5690
Here the P,I and D values are passed programatically to a class and error is calculated rather than measured.
"""

def apply_fitness_PID_program (gen, setPoint):
    global enable_program_pid
    enable_program_pid = 0

    # Iteration 1
    
    apply_fitness_PID_program.Kp = [0.4,0.8,1.2,1.6,2.0]
    apply_fitness_PID_program.Kd = [20.0,22.0,25.0,28.0,30.0]
    apply_fitness_PID_program.Ki = [0.1,0.5,1.0,2.0,4.0]
    apply_fitness_PID_program.e = [0] * 5
    random.shuffle(apply_fitness_PID_program.Kp)
    random.shuffle(apply_fitness_PID_program.Kd)
    random.shuffle(apply_fitness_PID_program.Ki)

    apply_fitness_PID_device.map1 = [{"Kp": kp, "Kd": kd, "Ki": ki, "e": er} 
                    for kp, kd, ki, er in zip(apply_fitness_PID_device.Kp, apply_fitness_PID_device.Kd, 
                                              apply_fitness_PID_device.Ki, apply_fitness_PID_device.e)]

    for i in range(5):
        pid = PIDController(Kp=apply_fitness_PID_device.map1[i]["Kp"], 
                Ki=apply_fitness_PID_device.map1[i]["Ki"], 
                Kd=apply_fitness_PID_device.map1[i]["Kd"], 
                setpoint=setPoint)
        control_output = pid.compute(setPoint, 1)
        print ("At step 1 iteration", i+1, "PID values and error", str(apply_fitness_PID_program.map1))
        apply_fitness_PID_device.map1[i]["e"] = abs(control_output - setPoint)/setPoint

    # Sort the P, I and D combinations based on minimum errors and apply selection (1st row) and crossover (2nd and 3rd row)
    
    apply_fitness_PID_program.map1.sort(key=lambda x: x["e"])
    t2 = apply_fitness_PID_program.map1[1]["Kd"]
    t3 = apply_fitness_PID_program.map1[1]["Ki"]
    apply_fitness_PID_program.map1[1]["Kd"] = apply_fitness_PID_program.map1[2]["Kd"]
    apply_fitness_PID_program.map1[1]["Ki"] = apply_fitness_PID_program.map1[2]["Ki"]
    apply_fitness_PID_program.map1[2]["Kd"] = t2
    apply_fitness_PID_program.map1[2]["Ki"] = t3
    t1 = apply_fitness_PID_program.map1[3]["Kp"]
    apply_fitness_PID_program.map1[3]["Kp"] = apply_fitness_PID_program.map1[1]["Kp"]
    apply_fitness_PID_program.map1[4]["Kp"] = t1
    apply_fitness_PID_program.map1[4]["Kd"] = t2
    apply_fitness_PID_program.map1[4]["Ki"] = t3

    # Iteration 2
    
    for i in range(5):
        pid = PIDController(Kp=apply_fitness_PID_device.map1[i]["Kp"], 
                Ki=apply_fitness_PID_device.map1[i]["Ki"], 
                Kd=apply_fitness_PID_device.map1[i]["Kd"], 
                setpoint=set_inverter_input)
        control_output = pid.compute(setPoint, 1)
        print ("At step 2 iteration", i+1, "PID values and error", str(apply_fitness_PID_program.map1))
        apply_fitness_PID_device.map1[i]["e"] = abs(control_output - setPoint)/setPoint

    # Sort the P, I and D combinations based on minimum errors and apply selection (1st row) and crossover (2nd and 3rd row)
    
    apply_fitness_PID_program.map1.sort(key=lambda x: x["e"])
    t2 = apply_fitness_PID_program.map1[1]["Kd"]
    t3 = apply_fitness_PID_program.map1[1]["Ki"]
    apply_fitness_PID_program.map1[1]["Kd"] = apply_fitness_PID_program.map1[2]["Kd"]
    apply_fitness_PID_program.map1[1]["Ki"] = apply_fitness_PID_program.map1[2]["Ki"]
    apply_fitness_PID_program.map1[2]["Kd"] = t2
    apply_fitness_PID_program.map1[2]["Ki"] = t3
    t1 = apply_fitness_PID_program.map1[3]["Kp"]
    apply_fitness_PID_program.map1[3]["Kp"] = apply_fitness_PID_program.map1[1]["Kp"]
    apply_fitness_PID_program.map1[4]["Kp"] = t1
    apply_fitness_PID_program.map1[4]["Kd"] = t2
    apply_fitness_PID_program.map1[4]["Ki"] = t3

    # Iteration 3
    
    for i in range(5):
        pid = PIDController(Kp=apply_fitness_PID_device.map1[i]["Kp"], 
                Ki=apply_fitness_PID_device.map1[i]["Ki"], 
                Kd=apply_fitness_PID_device.map1[i]["Kd"], 
                setpoint=set_inverter_input)
        control_output = pid.compute(setPoint, 1)
        print ("At step 3 iteration", i+1, "PID values and error", str(apply_fitness_PID_program.map1))
        apply_fitness_PID_device.map1[i]["e"] = abs(control_output - setPoint)/setPoint

    # Physically set the PID values
    
    apply_fitness_PID_program.map1.sort(key=lambda x: x["e"])
    getattr(mm[gen].getDevice(), 'setKp')(apply_fitness_PID_program.map1[0]["Kp"])
    time.sleep(0.2)
    getattr(mm[gen].getDevice(), 'setKd')(apply_fitness_PID_program.map1[0]["Kd"])
    time.sleep(0.2)
    getattr(mm[gen].getDevice(), 'setKi')(apply_fitness_PID_program.map1[0]["Ki"])
    time.sleep(0.2)


"""
Method to simulate the grid game for a single player. Objective is to calculate the utlitiy function 
only at discrete time intervals.
Mcjunkin, Timothy. (2021). Introducing the Grid Game. 10.1002/9781119660446.ch12
"""

def play_grid_game (windSupply, solarSupply):
    global game, power_demand, power_supply
    if game == 1:
        # Initialie battery storage and energy utilized in the grid to 0
        try:
            play_grid_game.battery_store
        except:
            play_grid_game.battery_store = 0

        try:
            play_grid_game.energy_utilized
        except:
            play_grid_game.energy_utilized = 0
            
        # Calculate maximum power in the grid for purchase
        player_supply = windSupply + solarSupply + float(power_input_in.value)
        set_inverter_input = float(inverter_power_in.value)
        prev_battery_state = play_grid_game.battery_store
        actual_player_demand = power_demand
        actual_player_supply = power_supply
        
        if player_supply >= (power_demand + float(inverter_power_in.value)):
            # Charge battery till full capacity is reached
            play_grid_game.battery_store = min(float(energy_storage_in.value), play_grid_game.battery_store + 
                                min(player_supply - (power_demand + float(inverter_power_in.value)), 
                                float(power_storage_in.value)) * (1/1800))
            actual_player_demand = power_demand + (play_grid_game.battery_store - prev_battery_state)*1800
            try:
                play_grid_game.prev_inverter_input
            except:
                play_grid_game.prev_inverter_input = player_supply - actual_player_demand
        else:
            # Discharge battery till 0 charge is stored
            play_grid_game.battery_store = max(0, play_grid_game.battery_store -  
                                min((power_demand + float(inverter_power_in.value)) - player_supply, 
                                float(power_storage_in.value)) * (1/1800))
            actual_player_supply = player_supply + (prev_battery_state - play_grid_game.battery_store)*1800
            try:
                play_grid_game.prev_inverter_input
            except:
                play_grid_game.prev_inverter_input = actual_player_supply - power_demand
            if play_grid_game.prev_inverter_input == 0:
                play_grid_game.prev_inverter_input = actual_player_supply - power_demand


        # Set pid control for the inverter input power
        pid = PIDController(Kp=0.5, Ki=0.001, Kd=0.001, setpoint=set_inverter_input)
        control_output = pid.compute(play_grid_game.prev_inverter_input, 2)
        # Remaining power is dissipated and wasted
        dissipation = play_grid_game.prev_inverter_input - control_output
        play_grid_game.prev_inverter_input = control_output

        # Calculate the utility function
        utility = 0
        if (actual_player_supply >= actual_player_demand):
            utility += (power_demand * float(unit_sale_price_in.value)) - (float(power_input_in.value) * float(unit_purchase_price_in.value))
        else:
            utility -= (actual_player_demand - actual_player_supply) * 0.7
        utility -= dissipation * 0.7
        utility -= abs(set_inverter_input - control_output) * 0.7

        
        # Cut the external power supply once the charge limit is reached
        play_grid_game.energy_utilized += float(power_input_in.value) * 1/1800
        if (play_grid_game.energy_utilized > float(energy_purchase_in.value)):
            power_input_in.value = str(0)

        print(actual_player_demand, actual_player_supply, control_output, dissipation, play_grid_game.battery_store,
              play_grid_game.energy_utilized, utility)

        # Display utility value at discrete time intervals
        utilityValue_Out.text = str(round(utility,6))

    elif game == 0:
        play_grid_game.battery_store = 0
        play_grid_game.energy_utilized = 0
        play_grid_game.prev_inverter_input = 0
        utilityValue_Out.text = str("0.000000")    

"""
Method to confirm selection of the command sent to the devices from dropdown list
"""

def dropdown_handler_selectCmd(cmd1, but1):
    def selectCmd_handler(event):
        cmd1.text = "<b>" + str(event.item) + "</b>"
        but1.disabled=False
    return selectCmd_handler

"""
Method to confirm the priority set for individual devices (either from demand or from supply perspective)
Priority is in the range of 1-5 and can be set only once till the stop button is pressed to reset the dashboard
The priority queue (heapq) in python implements a min heap, hence the values are negated.
"""

def button_handler_set_priority(key,priority,setPriority):
    def set_priority():
        p = 0
        try:
            if int(priority.value) >= 1 and int(priority.value) <= 5:
                p = -int(priority.value)
        except:
            pass
        if key not in pq_map:
            pq_map.add(key)
            if key.startswith('houseload'):
                heapq.heappush(pq_demand, (p, key))
            else:
                heapq.heappush(pq_supply, (p, key))
            print ("After setting priority queues:")
            print ("Demand :", pq_demand)
            print ("Supply :", pq_supply)
    return set_priority
        
widgets=[]

"""
Table listing history of the commands which were sent to respective devices - scrollable
"""

cmds = models.ColumnDataSource(data=dict(Device=[], Command=[], Value=[]))
columns = [
    models.TableColumn(field='Device', title='Device'),
    models.TableColumn(field='Command', title='Command'),
    models.TableColumn(field='Value', title='Value')
]
cmdString = ""
cmdCount = 0
data_table = models.DataTable(source=cmds, columns=columns, width=300, fit_columns=True)

"""
User input section - dropdown selection or text inputs
"""

for key in sorted(mm.keys()):
    if key.startswith('fan'):
        continue
    label = re.sub(r'\d+', '', key).upper()
    text = models.Div(text="<b>" + label + "</b>", width=100, height=30)
    cmdLst = [k for k in mm[key].getDevice().list_cmds().keys() if k != 'eoc' and not k.startswith('fan')]
    dropdown = models.Dropdown(width=150, height=30, label=key, menu=[(j, j) for j in cmdLst])
    inputCmd = models.Div(text="", width=100, height=30)
    inputVal = models.TextInput(width=100, height=30)
    button = models.widgets.Button(label="Apply", width=50, height=30)
    priority = models.TextInput(width=100, height=30)
    setPriority = models.widgets.Button(label="Priority(1-5)", width=80, height=30)
    button.disabled=True
    # button1 = models.widgets.Button(label="Control", width=50, height=30)
    
    dropdown.on_click(dropdown_handler_selectCmd(inputCmd, button))
    button.on_click(button_handler_apply(str(dropdown.label),inputCmd,inputVal,button))
    # button1.on_click(button_handler_control(button1))
    setPriority.on_click(button_handler_set_priority(key,priority,setPriority))
    
    widgets.append(row(text, dropdown, inputCmd, inputVal, button, priority, setPriority))
 
dropdown_layout = column(*widgets)
data_table.height = max(100, len(mm.keys()) * 40)

spacer_width = models.Spacer(width=50, height=30)
spacer_height = models.Spacer(width=50, height=30)

"""
The window defines the time range for which the plots will display the results
"""

def setWindow (attr, old, new):
    window_universal = new

slide_window = models.widgets.Slider(start=20, end=500, value=80, step=20, title="Sliding Window Time Range")
slide_window.on_change('value', setWindow)

"""
This method is used to set the rotation speed of the fans connected to the windmills
"""

def setFans (attr, old, new):
    fanSpeed = new
    oldfanSpeed = old
    for j, fan in enumerate([x for x in mm.keys() if x.startswith('fan')]):
        with suppress_stdout():
            try:
                if fanSpeed == 0:
                    mm[fan].getDevice().call('fanOff')
                elif oldfanSpeed == 0:
                    mm[fan].getDevice().call('fanOn')
                    getattr(mm[fan].getDevice(), 'setSpeed')(int(fanSpeed))
                else:
                    getattr(mm[fan].getDevice(), 'setSpeed')(int(fanSpeed))
            except:
                pass

slide_fan = models.widgets.Slider(start=0, end=200, value=10, step=10, title="Fans Control")
slide_fan.on_change('value', setFans)

"""
The stop button resets the devices by calling the trackOff or off commands for respective devices 
and clears the priority queues.
"""

def button_stop_handler(buttonStop):
    def stop_handler():
        global pq_demand, pq_supply, pq_map
        for j, gen in enumerate(mm.keys()):
            if gen.startswith('generator') or gen.startswith('solar') or gen.startswith('wind'):
                mm[gen].getDevice().call("trackOff")
            elif gen.startswith('fan'):
                mm[gen].getDevice().call("fanOff")
            else:
                mm[gen].getDevice().call("off") 
        pq_demand = []
        pq_supply = []
        pq_map.clear()
    return stop_handler

buttonStop = models.widgets.Button(label="Stop", width=50, height=30)
buttonStop.on_click(button_stop_handler(buttonStop))

"""
This method does supply side power balancing by turning off generators if demanded power is reached
"""

def button_bal_supply_handler(buttonBalSupply):
    def bal_supply_handler():
        global pq_demand, pq_supply, pq_map
        global power_demand
        print("Balancing Supply")
        temp = []
        prev = -6
        total = 0
        heapq.heappush(pq_supply,(1,'generator_eof'))
        while (pq_supply):
            q = heapq.heappop(pq_supply)
            dev_pri = q[0]
            dev_name = q[1]
            print("Acting on balancing for", dev_name)
            if not dev_name.startswith('generator'):
                with suppress_stdout():
                    outLoad = 0
                    try:
                        outLoad = float(mm[dev_name].getDevice().call('getKW')[0])
                    except:
                        pass
                    outLoad = 0 if math.isnan(outLoad) else outLoad
                total += outLoad
                pq_map.remove(dev_name)
            else:
                if dev_pri != prev:
                    j = 0
                    for dev in temp:
                        print("Balancing for", dev, "with demand", power_demand, "and current power total", total, "length", len(temp))
                        if total < power_demand:
                            mm[dev].getDevice().call('trackOn')
                            time.sleep(0.2)
                            getattr(mm[dev].getDevice(), 'setLoad')(min((int(power_demand) - int(total))//len(temp), gen_max))
                            print("Setting load output of", dev, "to", str(min((int(power_demand) - int(total))//len(temp), gen_max)))
                            j += min((int(power_demand) - int(total))//len(temp), gen_max)
                        else:
                            mm[dev].getDevice().call('trackOff')
                        pq_map.remove(dev)
                    total += j
                    temp = []
                prev = dev_pri
                temp.append(dev_name)
            
    return bal_supply_handler

"""
This method does demand side power balancing by turning off houseloads if supply power is exceeded
"""

def button_bal_demand_handler(buttonBalDemand):
    def bal_demand_handler():
        global pq_demand, pq_supply, pq_map
        global power_supply
        total_consumption = 0
        print("Balancing Demand")
        while (pq_demand):
            q = heapq.heappop(pq_demand)
            dev_pri = q[0]
            dev_name = q[1]
            pq_map.remove(dev_name)
            outLoad = household_get_call(mm[dev_name].getDevice(), 'getLoadVal')
            total_consumption += outLoad
            if total_consumption > power_supply:
                mm[dev_name].getDevice().call('lightsOut')
                
    return bal_demand_handler

buttonBalSupply = models.widgets.Button(label="Balance Supply", width=120, height=30)
buttonBalSupply.on_click(button_bal_supply_handler(buttonBalSupply))

buttonBalDemand = models.widgets.Button(label="Balance Demand", width=120, height=30)
buttonBalDemand.on_click(button_bal_demand_handler(buttonBalDemand))

layout_with_tab = row(dropdown_layout, spacer_width, data_table)
io.curdoc().add_root(layout_with_tab)

io.curdoc().add_root(row(slide_window, buttonStop, spacer_width, buttonBalSupply, buttonBalDemand))
io.curdoc().add_root(slide_fan)

p = plotting.figure(
    x_axis_label="Time", y_axis_label="OutputLoad", title="Generator(s) Output Load (kW)",
    width=550, height=300, x_axis_type="linear",
)
p.add_layout(models.Legend(), 'right')
q = plotting.figure(
    x_axis_label="Time", y_axis_label="OutputLoad", title="Solar Tracker(s) Output Load (kW)",
    width=550, height=300, x_axis_type="linear",
)
q.add_layout(models.Legend(), 'right')
r = plotting.figure(
    x_axis_label="Time", y_axis_label="OutputLoad", title="Wind Turbine(s) Output Load (kW)",
    width=550, height=300, x_axis_type="linear",
)
r.add_layout(models.Legend(), 'right')

pv = plotting.figure(
    x_axis_label="Time", y_axis_label="VoltageOutput", title="Generator(s) Output Voltage (V)",
    width=550, height=300, x_axis_type="linear",
)
pv.add_layout(models.Legend(), 'right')
qv = plotting.figure(
    x_axis_label="Time", y_axis_label="VoltageOutput", title="Solar Tracker(s) Output Voltage (V)",
    width=550, height=300, x_axis_type="linear",
)
qv.add_layout(models.Legend(), 'right')
rv = plotting.figure(
    x_axis_label="Time", y_axis_label="VoltageOutput", title="Wind Turbine(s) Output Voltage (V)",
    width=550, height=300, x_axis_type="linear",
)
rv.add_layout(models.Legend(), 'right')

s = plotting.figure(
    x_axis_label="Time", y_axis_label="HouseLoads", title="Power Demand (kW)",
    width=400, height=300, x_axis_type="linear",
)
t = plotting.figure(
    x_axis_label="Time", y_axis_label="CO2Accumulation", title="Cumulative CO2 Aggregated (Tons)",
    width=400, height=300, x_axis_type="linear",
)
u = create_dial()

plotGenerators()
plotSolars()
plotWindTurbines()
plotDemandAndCO2()
# plotGauge()

io.curdoc().add_root(row(p, q, r))
io.curdoc().add_root(row(pv, qv, rv))
io.curdoc().add_root(row(spacer_width, spacer_width, spacer_width, s, t, u))

def button_gen0_pid_device():
    global pid_iter0
    if 'generator0' in mm:
        pid_iter0 = 1
        mm['generator0'].getDevice().call('trackOn')
        print("Setting Generator 0 to", str(int(setGenLoad.value)))
        getattr(mm['generator0'].getDevice(), 'setLoad')(int(setGenLoad.value))

def button_gen1_pid_program():
    global pid_iter1
    if 'generator1' in mm:
        pid_iter1 = 1
        mm['generator1'].getDevice().call('trackOn')
        print("Setting Generator 1 to", str(int(setGenLoad.value)))
        getattr(mm['generator1'].getDevice(), 'setLoad')(int(setGenLoad.value))

io.curdoc().add_root(spacer_height)
exp = models.Div(text="<b>EXPERIMENTAL</b>", width=200, height=30)
io.curdoc().add_root(exp)

setGenDiv = models.Div(text="Set Load for Generators", width=250, height=30)
setGenLoad = models.TextInput(width=100, height=30)
setGen0LoadApply = models.widgets.Button(label="Device's PID control Gen0", width=200, height=30)
setGen0LoadApply.on_click(button_gen0_pid_device)
setGen1LoadCalc = models.widgets.Button(label="Programmed PID control Gen1", width=200, height=30)
setGen1LoadCalc.on_click(button_gen1_pid_program)

io.curdoc().add_root(spacer_height)
exp1 = models.Div(text="<b>PID CONTROL FOR GENERATOR POWER</b>", width=250, height=30)
io.curdoc().add_root(exp1)
io.curdoc().add_root(row(setGenDiv, setGenLoad, setGen0LoadApply, setGen1LoadCalc))

energy_purchase = models.Div(text="<b>Energy Purchased (kWh)</b>", width=160, height=30)
energy_purchase_in = models.TextInput(width=160, height=30)
power_input = models.Div(text="<b>Power Input (kW)</b>", width=160, height=30)
power_input_in = models.TextInput(width=160, height=30)
unit_purchase_price = models.Div(text="<b>Unit Purchase Price ($)</b>", width=160, height=30)
unit_purchase_price_in = models.TextInput(width=160, height=30)
unit_sale_price = models.Div(text="<b>Unit Sale Price ($)</b>", width=160, height=30)
unit_sale_price_in = models.TextInput(width=160, height=30)
energy_storage = models.Div(text="<b>Storage Energy Limit (kWh)</b>", width=160, height=30)
energy_storage_in = models.TextInput(width=160, height=30)
power_storage = models.Div(text="<b>Storage Power Limit (kW)</b>", width=160, height=30)
power_storage_in = models.TextInput(width=160, height=30)
inverter_power = models.Div(text="<b>Inverter Power for 60 Hz</b>", width=160, height=30)
inverter_power_in = models.TextInput(width=160, height=30)
gridGame = models.widgets.Button(label="PlayGame", width=120, height=30)
def button_gridGame():
    global game
    game = 1
gridGame.on_click(button_gridGame)
stopGame = models.widgets.Button(label="StopGame", width=120, height=30)
def button_stopGame():
    global game
    game = 0
stopGame.on_click(button_stopGame)
utilityValue = models.Div(text="<b>Utility</b>", width=160, height=30)
utilityValue_Out = models.Div(text="", width=160, height=30)

io.curdoc().add_root(spacer_height)
exp2 = models.Div(text="<b>GRID GAME</b>", width=250, height=30)
io.curdoc().add_root(exp2)
io.curdoc().add_root(row(energy_purchase, power_input, unit_purchase_price, unit_sale_price, energy_storage, power_storage, 
                         inverter_power, utilityValue))
io.curdoc().add_root(row(energy_purchase_in, power_input_in, unit_purchase_price_in, unit_sale_price_in, energy_storage_in, power_storage_in,
                         inverter_power_in, utilityValue_Out))
io.curdoc().add_root(row(gridGame, stopGame))

io.curdoc().add_periodic_callback(update, 2000)

In [ ]:
#For operating through macbook local host

!bokeh serve streaming

In [ ]:
#For operating through raspberry pi via macbook

!python3 -m bokeh  serve streaming --allow-websocket-origin='*' --address 10.55.0.1 --port 5006